# Alt Signals Daily Refresh (yfinance)
Incremental ingestion of alternative market signals for `company_universe` symbols.

**Outputs**:
- `{catalog}.gold.analyst_recommendations`
- `{catalog}.gold.options_iv_skew_daily`
- `{catalog}.gold.short_interest_snapshot`
- `{catalog}.gold.insider_form4`

**Source**: Yahoo Finance (free, no API key)

**Schedule**: Daily after market close (7:30 PM ET)

**Parameters**:
| Widget | Default | Description |
|--------|---------|-------------|
| `catalog` | riskbricks | Unity Catalog name |
| `as_of_date` | (auto) | Yesterday ET |
| `max_symbols` | 0 | 0=all from company_universe |
| `sleep_seconds` | 0.3 | Throttle between API calls |

In [0]:
%pip install yfinance --quiet
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "riskbricks", "Catalog Name")
dbutils.widgets.text("as_of_date", "", "As of Date (YYYY-MM-DD, blank=yesterday)")
dbutils.widgets.text("max_symbols", "0", "Max Symbols (0=all)")
dbutils.widgets.text("sleep_seconds", "0.3", "Sleep Between API Calls")

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import time
import json
import pandas as pd
import yfinance as yf
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
)

# -- Read widgets -------------------------------------------------------
catalog = dbutils.widgets.get("catalog").strip()
local_tz = ZoneInfo("America/New_York")

as_of_input = dbutils.widgets.get("as_of_date").strip()
if as_of_input:
    as_of_date = as_of_input
else:
    as_of_date = (datetime.now(local_tz).date() - timedelta(days=1)).strftime("%Y-%m-%d")

max_symbols = int(dbutils.widgets.get("max_symbols") or "0")
sleep_seconds = float(dbutils.widgets.get("sleep_seconds") or "0.3")

# -- Ensure schemas exist -----------------------------------------------
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

# -- Load symbols from company_universe ---------------------------------
symbols = [
    row.symbol
    for row in spark.sql(
        f"SELECT DISTINCT symbol FROM {catalog}.gold.company_universe ORDER BY symbol"
    ).collect()
]
if max_symbols and max_symbols > 0:
    symbols = symbols[:max_symbols]

print(f"Config: catalog={catalog}, as_of_date={as_of_date}")
print(f"Symbols: {len(symbols)} (max_symbols={max_symbols})")
print(f"Sleep: {sleep_seconds}s between calls")

In [0]:
def safe_info(ticker):
    try:
        return ticker.info or {}
    except Exception:
        return {}

def _safe_numeric(val):
    if pd.isna(val):
        return None
    try:
        return float(val)
    except Exception:
        return None

def _safe_date(val):
    if pd.isna(val):
        return None
    try:
        if hasattr(val, "to_pydatetime"):
            return val.to_pydatetime()
        elif hasattr(val, "date"):
            return datetime.combine(val.date(), datetime.min.time())
        return pd.to_datetime(val).to_pydatetime() if pd.notna(val) else None
    except Exception:
        return None

analyst_rows = []
options_rows = []
short_rows = []
insider_rows = []

succeeded = 0
failed = 0

for i, sym in enumerate(symbols):
    try:
        t = yf.Ticker(sym)
        info = safe_info(t)
        if not info or len(info) < 5:
            failed += 1
            continue

        # -- Analyst recommendations ------------------------------------
        try:
            recs = t.recommendations
            if recs is not None and not recs.empty:
                recs = recs.reset_index()
                for _, row in recs.iterrows():
                    analyst_rows.append({
                        "symbol": sym,
                        "event_date": _safe_date(row.get("Date", row.get("date"))),
                        "firm": str(row.get("Firm", row.get("firm", ""))),
                        "to_grade": str(row.get("To Grade", row.get("toGrade", ""))),
                        "from_grade": str(row.get("From Grade", row.get("fromGrade", ""))),
                        "action": str(row.get("Action", row.get("action", ""))),
                        "as_of_date": as_of_date,
                    })
        except Exception:
            pass

        # -- Options IV skew --------------------------------------------
        try:
            expirations = t.options
            if expirations:
                for exp in expirations[:3]:  # first 3 expirations
                    chain = t.option_chain(exp)
                    call_iv = chain.calls["impliedVolatility"].mean() if not chain.calls.empty else None
                    put_iv = chain.puts["impliedVolatility"].mean() if not chain.puts.empty else None
                    skew = (put_iv - call_iv) if (put_iv and call_iv) else None
                    options_rows.append({
                        "symbol": sym,
                        "expiration": exp,
                        "call_iv": _safe_numeric(call_iv),
                        "put_iv": _safe_numeric(put_iv),
                        "iv_skew": _safe_numeric(skew),
                        "as_of_date": as_of_date,
                    })
        except Exception:
            pass

        # -- Short interest ---------------------------------------------
        try:
            short_ratio = _safe_numeric(info.get("shortRatio"))
            short_pct = _safe_numeric(info.get("shortPercentOfFloat"))
            shares_short = _safe_numeric(info.get("sharesShort"))
            shares_prior = _safe_numeric(info.get("sharesShortPriorMonth"))
            if any(v is not None for v in [short_ratio, short_pct, shares_short]):
                short_rows.append({
                    "symbol": sym,
                    "short_ratio": short_ratio,
                    "short_percent_float": short_pct,
                    "shares_short": shares_short,
                    "shares_short_prior": shares_prior,
                    "short_interest": shares_short,
                    "as_of_date": as_of_date,
                })
        except Exception:
            pass

        # -- Insider transactions (Form 4) ------------------------------
        try:
            insiders = t.insider_transactions
            if insiders is not None and not insiders.empty:
                for _, row in insiders.iterrows():
                    insider_rows.append({
                        "symbol": sym,
                        "filing_date": _safe_date(row.get("Start Date", row.get("startDate"))),
                        "insider_name": str(row.get("Insider", row.get("insider", ""))),
                        "title": str(row.get("Position", row.get("position", ""))),
                        "transaction_type": str(row.get("Transaction", row.get("transaction", ""))),
                        "shares": _safe_numeric(row.get("Shares", row.get("shares"))),
                        "value": _safe_numeric(row.get("Value", row.get("value"))),
                        "as_of_date": as_of_date,
                    })
        except Exception:
            pass

        succeeded += 1
    except Exception as e:
        failed += 1

    if (i + 1) % 25 == 0:
        print(f"  Progress: {i+1}/{len(symbols)} ({succeeded} ok, {failed} failed)")
    time.sleep(sleep_seconds)

print(f"\nFetch complete: {succeeded} succeeded, {failed} failed")
print(f"  Analyst recs: {len(analyst_rows)}")
print(f"  Options IV:   {len(options_rows)}")
print(f"  Short int:    {len(short_rows)}")
print(f"  Insider txns: {len(insider_rows)}")

In [0]:
def write_gold_table(table_name, rows, schema):
    """Append new data to gold table, partitioned by as_of_date + symbol."""
    if not rows:
        print(f"  No rows for {table_name} -- skipping")
        return 0
    df = spark.createDataFrame(rows, schema=schema)
    df = df.withColumn("ingestion_timestamp", F.current_timestamp())
    if not spark.catalog.tableExists(table_name):
        df.write.mode("overwrite").partitionBy("as_of_date", "symbol").saveAsTable(table_name)
    else:
        # Delete existing data for this as_of_date, then append (idempotent re-run)
        spark.sql(f"DELETE FROM {table_name} WHERE as_of_date = '{as_of_date}'")
        df.write.mode("append").saveAsTable(table_name)
    cnt = df.count()
    print(f"  Saved {cnt} rows to {table_name}")
    return cnt

analyst_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("event_date", TimestampType(), True),
    StructField("firm", StringType(), True),
    StructField("to_grade", StringType(), True),
    StructField("from_grade", StringType(), True),
    StructField("action", StringType(), True),
    StructField("as_of_date", StringType(), False),
])

options_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("expiration", StringType(), True),
    StructField("call_iv", DoubleType(), True),
    StructField("put_iv", DoubleType(), True),
    StructField("iv_skew", DoubleType(), True),
    StructField("as_of_date", StringType(), False),
])

short_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("short_ratio", DoubleType(), True),
    StructField("short_percent_float", DoubleType(), True),
    StructField("shares_short", DoubleType(), True),
    StructField("shares_short_prior", DoubleType(), True),
    StructField("short_interest", DoubleType(), True),
    StructField("as_of_date", StringType(), False),
])

insider_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("filing_date", TimestampType(), True),
    StructField("insider_name", StringType(), True),
    StructField("title", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("shares", DoubleType(), True),
    StructField("value", DoubleType(), True),
    StructField("as_of_date", StringType(), False),
])

a_cnt = write_gold_table(f"{catalog}.gold.analyst_recommendations", analyst_rows, analyst_schema)
o_cnt = write_gold_table(f"{catalog}.gold.options_iv_skew_daily", options_rows, options_schema)
s_cnt = write_gold_table(f"{catalog}.gold.short_interest_snapshot", short_rows, short_schema)
i_cnt = write_gold_table(f"{catalog}.gold.insider_form4", insider_rows, insider_schema)

In [0]:
result = {
    "status": "success",
    "catalog": catalog,
    "as_of_date": as_of_date,
    "symbols_processed": succeeded,
    "symbols_failed": failed,
    "analyst_recommendations": a_cnt,
    "options_iv_skew": o_cnt,
    "short_interest": s_cnt,
    "insider_form4": i_cnt,
}

print("=" * 60)
print("ALT SIGNALS DAILY REFRESH COMPLETE")
print("=" * 60)
print(f"  Catalog:      {catalog}")
print(f"  As of date:   {as_of_date}")
print(f"  Symbols:      {succeeded} ok / {failed} failed")
print(f"  Analyst recs: {a_cnt:,}")
print(f"  Options IV:   {o_cnt:,}")
print(f"  Short int:    {s_cnt:,}")
print(f"  Insider txns: {i_cnt:,}")

dbutils.notebook.exit(json.dumps(result))